# Creating Cherry Rainbow Tables

For N = 2**16, currently only making one table

## Imports

In [23]:
import pickle
import random
from hashlib import sha256
import mmh3
from tqdm import tqdm
from math import pi, sqrt, e, log

## Table Parameters

### Initialise Startpoints

In [24]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

### Parameters

In [25]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_target = N**(2/3) # our target mt
m_0 = round(mt_target/(1-alpha))    # m_0 - number of startpoints
##############################################################################
# initialise startpoints 
startpoints = get_startpoints(N, m_0, nlabel, alpha)


### Cherry-picks per column - Kis

In [26]:
# get # cherry-picks per column from pickle file 

# load pickle file
with open(f'higher_costs_ftol_1.pickle', 'rb') as f:
    data = pickle.load(f)

# structure of file
# array of different alpha used
    # for each alpha, array of different costs used
    # for each cost, array arranged as [Kjs, m_0, final_cost, m_values]

# to get Kjs for alpha = 0.95 and cost factor = 25
Kis = data[-1][4][0]

## Hash and Reduction Functions

In [27]:
# Hash function
def H(x):
	return int(sha256(bytes(x)).hexdigest(), 16)

# Reduction function
# currently mod but should change to murmurhash in future
def r(y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
	return (y + i + ell*t) % N

## Building the Table

In [40]:
# take in m_0 and t as parameters - how many chains to start with and how long to make the chains
# store the table as a dictionary of endpoint:startpoint pairs (rather than sp:ep for easier lookup later)
# store in a pickle file
# need to also store the reduction function used in each column - how much more memory is taken up? only need to store index per column, so t more bits of memory

def build_cherry_table(t, alpha, startpoints, Kis):
    # store table in dictionary
    table = {}  # store all the points then remove duplicate entries - can't do duplicate keys in dictionary anyway so we can just store all ep:sp

    # Instead of making the table chain by chain, we have to make it column by column to test what reduction function to choose
    # store reduction function indexes
    rf_indexes = []

    # initialise table wit SP:SP pairs
    for point in startpoints:
        table[point] = point

    # hash all current points
    hashed_points = [H(sp) for sp in startpoints]    

    """
    - for each column
        - hash the current points
        - get the # cherry-picks for that column
        - use each reduction function on the points, and store the best running m_i+1 and reduction function index
            - is it quicker to store the reduced points and overwrite them each time, or tally the points as we test and then reduce all points at the end with best r - but then we're doing m_i more reductions?
        - for each rf sample:
            - reduce the column and store in a set
            - get the length of the set - if its better than before then note down this rf 
            - clear the set and do it again
        - with our best rf, 
    """

    # for each column in the table
    for i in range(t):

        # get # cherry-picks for this column
        k_i = round(Kis[i])

        # variables to store best reduction function info
        best_m_i_plus_1 = -1
        best_rf_index = -1

        # set to store reduced points for each rf test
        reduced_points_set = set()

        # test each reduction function
        for rf_index in range(k_i):
            # reduce all hashed points with this reduction function
            for hp in hashed_points:
                rp = r(hp, rf_index)  # reduce point
                reduced_points_set.add(rp)   # add to set

            # get m_i+1
            m_i_plus_1 = len(reduced_points_set)

            # check if best
            if m_i_plus_1 > best_m_i_plus_1:
                best_m_i_plus_1 = m_i_plus_1
                best_rf_index = rf_index

            # clear set for next rf test
            reduced_points_set.clear()

        # with best rf, reduce all hashed points and update for next column

        # make a current table to now have m_i:SP pairs
        current_column = dict()
        # for each (key, value) pair in the current table, make its new entry
            # i don't want to hash again - can i reuse hashes_points? how do I know its in order?
        for (key, sp), hp in zip(table.items(), hashed_points):
            ep = r(hp, best_rf_index)  # get endpoint with best rf
            current_column[ep] = sp  # store m_i:SP pair

        # update table to current column
        table = current_column

        # update hashed_points
        hashed_points = [H(m_i) for m_i in table.keys()]

        # store the best rf index
        rf_indexes.append(best_rf_index)
        
    # store the table and rf_indexes in a pickle file
    # with open(f'cherry_table_alpha_{alpha}_t_{t}.pkl', 'wb') as f:
    #     pickle.dump((table), f)

    # with open(f'cherry_rf_alpha_{alpha}_t_{t}.pkl', 'wb') as f:
    #     pickle.dump((table), f)


    return table

## Run

### Precomputation Phase - Build the table

In [41]:
# either build or load table
def get_cherry_table():
    # try loading table from pickle file
    try:
        with open(f'cherry_table_alpha_{alpha}_t_{t}.pkl', 'rb') as f:
            table = pickle.load(f)

    # if no pickle file found, build the table
    except FileNotFoundError:
        table = build_cherry_table(t, alpha, startpoints, Kis)

    return table


In [ ]:
table = get_cherry_table()

In [31]:
table

{}